In [2]:
# Cell 1 — connect Colab to Earth Engine
import ee

# This pops up a Google sign-in. Use the SAME Google account as your Earth Engine.
ee.Authenticate()

# Use your Earth Engine project (the one from the GEE editor)
ee.Initialize(project='isu-lceuranie')

print('Connected to Earth Engine ✓')

Connected to Earth Engine ✓


In [3]:
# Cell 2 — methane reading at Groundbirch (same as the GEE editor)

# 1. Facility pin
facility = ee.Geometry.Point([-121.077735, 55.962807])

# 2. Areas built from the pin
aoi        = facility.buffer(35000).bounds()   # analysis box
site_zone  = facility.buffer(7000)             # the site
background = facility.buffer(35000).difference(facility.buffer(15000))  # ring around it

# 3. Average TROPOMI methane over the period
start, end = '2026-03-01', '2026-06-01'
ch4 = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CH4')
       .select('CH4_column_volume_mixing_ratio_dry_air')
       .filterDate(start, end)
       .filterBounds(aoi)
       .mean())

# 4. Helper: average methane inside any shape
def mean_ch4(region):
    return ch4.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region, scale=1000, maxPixels=1e9
    ).getNumber('CH4_column_volume_mixing_ratio_dry_air').getInfo()

# 5. The numbers
site_val = mean_ch4(site_zone)
bg_val   = mean_ch4(background)
excess   = site_val - bg_val
pct_over = excess / bg_val * 100

print(f'Site methane:       {site_val:.2f} ppb')
print(f'Background methane: {bg_val:.2f} ppb')
print(f'Excess at site:     {excess:.2f} ppb')
print(f'Percent above bkgd: {pct_over:.3f} %')

Site methane:       1862.95 ppb
Background methane: 1864.99 ppb
Excess at site:     -2.04 ppb
Percent above bkgd: -0.109 %


In [4]:
# Cell 3 — NO2 and CO at Groundbirch

def mean_over_aoi(collection_id, band):
    img = (ee.ImageCollection(collection_id)
           .select(band)
           .filterDate(start, end)
           .filterBounds(aoi)
           .mean())
    return img.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=1000, maxPixels=1e9
    ).getNumber(band).getInfo()

no2 = mean_over_aoi('COPERNICUS/S5P/OFFL/L3_NO2', 'tropospheric_NO2_column_number_density')
co  = mean_over_aoi('COPERNICUS/S5P/OFFL/L3_CO',  'CO_column_number_density')

print(f'NO2: {no2:.8f} mol/m2')
print(f'CO:  {co:.5f} mol/m2')

NO2: 0.00001153 mol/m2
CO:  0.02797 mol/m2


In [5]:
# Cell 4 — the reusable engine: analyze any facility from its coordinates

def analyze_facility(lat, lon, start='2026-03-01', end='2026-06-01'):
    pin   = ee.Geometry.Point([lon, lat])
    aoi   = pin.buffer(35000).bounds()
    site  = pin.buffer(7000)
    bkgd  = pin.buffer(35000).difference(pin.buffer(15000))

    ch4 = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CH4')
           .select('CH4_column_volume_mixing_ratio_dry_air')
           .filterDate(start, end).filterBounds(aoi).mean())

    def m(img, region, band):
        return img.reduceRegion(ee.Reducer.mean(), region, 1000, maxPixels=1e9).getNumber(band).getInfo()

    band = 'CH4_column_volume_mixing_ratio_dry_air'
    site_ch4 = m(ch4, site, band)
    bg_ch4   = m(ch4, bkgd, band)

    def gas(cid, b):
        img = ee.ImageCollection(cid).select(b).filterDate(start, end).filterBounds(aoi).mean()
        return m(img, aoi, b)

    return {
        'lat': lat, 'lon': lon,
        'methane_site_ppb':   round(site_ch4, 2),
        'methane_bkgd_ppb':   round(bg_ch4, 2),
        'methane_excess_pct': round((site_ch4 - bg_ch4) / bg_ch4 * 100, 3),
        'no2_mol_m2': gas('COPERNICUS/S5P/OFFL/L3_NO2', 'tropospheric_NO2_column_number_density'),
        'co_mol_m2':  gas('COPERNICUS/S5P/OFFL/L3_CO',  'CO_column_number_density'),
    }

# Run it for Groundbirch
result = analyze_facility(55.962807, -121.077735)
result

{'lat': 55.962807,
 'lon': -121.077735,
 'methane_site_ppb': 1862.95,
 'methane_bkgd_ppb': 1864.99,
 'methane_excess_pct': -0.109,
 'no2_mol_m2': 1.1528536208611025e-05,
 'co_mol_m2': 0.02796838491574994}

In [6]:
# Cell 5 — flaring from the uploaded EOG Excel
import pandas as pd

flares = pd.read_excel('VIIRS_2024.xlsx')

def flaring_bcm(lat, lon, box=0.1):
    near = flares[
        (flares['Latitude'].between(lat - box, lat + box)) &
        (flares['Longitude'].between(lon - box, lon + box))
    ]
    return round(near['BCM 2024'].sum(), 5)

print('Flaring near Groundbirch (BCM/yr):', flaring_bcm(55.962807, -121.077735))

Flaring near Groundbirch (BCM/yr): 0.0


In [7]:
# Cell 6 — monthly methane trajectory (the line for the chart)
import datetime

def monthly_methane(lat, lon, months_back=12):
    pin  = ee.Geometry.Point([lon, lat])
    aoi  = pin.buffer(35000).bounds()
    band = 'CH4_column_volume_mixing_ratio_dry_air'

    # walk back month by month from today
    today = datetime.date.today().replace(day=1)
    series = []
    for i in range(months_back, 0, -1):
        # first day of the target month, and of the next month
        m_start = (today - datetime.timedelta(days=31*i)).replace(day=1)
        m_end   = (m_start + datetime.timedelta(days=32)).replace(day=1)

        img = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CH4')
               .select(band)
               .filterDate(str(m_start), str(m_end))
               .filterBounds(aoi)
               .mean())

        # average methane over the site that month (None if no clear passes)
        val = img.reduceRegion(ee.Reducer.mean(), pin.buffer(7000), 1000, maxPixels=1e9) \
                 .get(band).getInfo()

        series.append({'month': m_start.strftime('%Y-%m'),
                       'methane_ppb': round(val, 2) if val is not None else None})
    return series

trajectory = monthly_methane(55.962807, -121.077735)
for row in trajectory:
    print(row['month'], '→', row['methane_ppb'], 'ppb' if row['methane_ppb'] else '(no data)')

2025-05 → 1875.04 ppb
2025-06 → 1839.25 ppb
2025-07 → 1855.54 ppb
2025-08 → 1871.21 ppb
2025-09 → 1861.21 ppb
2025-10 → 1901.32 ppb
2025-11 → None (no data)
2025-12 → None (no data)
2026-01 → None (no data)
2026-02 → 1900.01 ppb
2026-03 → 1865.24 ppb
2026-05 → 1862.5 ppb


In [8]:
# Korpezhe, Turkmenistan — documented super-emitter (Varon et al. 2019)
korpezhe = analyze_facility(38.499, 54.199)
korpezhe

{'lat': 38.499,
 'lon': 54.199,
 'methane_site_ppb': 1972.22,
 'methane_bkgd_ppb': 1960.0,
 'methane_excess_pct': 0.624,
 'no2_mol_m2': 1.686045977798825e-05,
 'co_mol_m2': 0.03188264804535826}

In [9]:
# Korpezhe trajectory + flaring (same functions, new coordinates)
korpezhe_trajectory = monthly_methane(38.499, 54.199)
korpezhe_flaring    = flaring_bcm(38.499, 54.199)

print('Korpezhe flaring (BCM/yr):', korpezhe_flaring)
for row in korpezhe_trajectory:
    print(row['month'], '→', row['methane_ppb'], 'ppb' if row['methane_ppb'] else '(no data)')

Korpezhe flaring (BCM/yr): 0.0093
2025-05 → 1970.4 ppb
2025-06 → 1968.79 ppb
2025-07 → 1990.75 ppb
2025-08 → 2000.84 ppb
2025-09 → 2014.04 ppb
2025-10 → 1996.91 ppb
2025-11 → 2002.79 ppb
2025-12 → 2013.87 ppb
2026-01 → 1967.31 ppb
2026-02 → 1963.88 ppb
2026-03 → 1977.27 ppb
2026-05 → 1977.92 ppb


In [10]:
# Region method: compare a basin to a distant clean reference
def analyze_area(name, lat, lon, ref_lat, ref_lon, radius_km=40,
                 start='2026-03-01', end='2026-06-01'):
    band = 'CH4_column_volume_mixing_ratio_dry_air'
    ch4  = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CH4').select(band).filterDate(start, end)
    def mean_at(plat, plon):
        region = ee.Geometry.Point([plon, plat]).buffer(radius_km*1000)
        return ch4.filterBounds(region).mean() \
                  .reduceRegion(ee.Reducer.mean(), region, 1000, maxPixels=1e9) \
                  .getNumber(band).getInfo()
    target = mean_at(lat, lon)
    ref    = mean_at(ref_lat, ref_lon)
    return {'name': name, 'lat': lat, 'lon': lon,
            'target_ch4_ppb': round(target, 2),
            'reference_ch4_ppb': round(ref, 2),
            'enhancement_pct': round((target - ref) / ref * 100, 3)}

# Permian (Delaware sub-basin) vs a clean central-Arizona desert reference
permian = analyze_area('Permian Basin (Delaware)', 31.9, -103.9,
                       ref_lat=34.5, ref_lon=-111.5)
permian

{'name': 'Permian Basin (Delaware)',
 'lat': 31.9,
 'lon': -103.9,
 'target_ch4_ppb': 1929.64,
 'reference_ch4_ppb': 1884.22,
 'enhancement_pct': 2.411}

In [11]:
permian_trajectory = monthly_methane(31.9, -103.9)
permian_flaring    = flaring_bcm(31.9, -103.9)
print('Permian flaring (BCM/yr):', permian_flaring)
for row in permian_trajectory:
    print(row['month'], '→', row['methane_ppb'], 'ppb' if row['methane_ppb'] else '(no data)')

Permian flaring (BCM/yr): 0.08877
2025-05 → 1929.38 ppb
2025-06 → 1920.89 ppb
2025-07 → 1916.76 ppb
2025-08 → 1933.97 ppb
2025-09 → 1953.19 ppb
2025-10 → 1952.93 ppb
2025-11 → 1954.13 ppb
2025-12 → 1948.06 ppb
2026-01 → 1933.72 ppb
2026-02 → 1930.16 ppb
2026-03 → 1926.03 ppb
2026-05 → 1932.69 ppb


In [12]:
# Final cell — package all three facilities into one file for the website
import json, datetime

def package(name, operator, region, kind, snapshot, trajectory, flaring,
            excess_field, excess_value):
    return {
        'name': name, 'operator': operator, 'region': region,
        'method': kind,                       # 'facility' or 'basin'
        'is_reference_site': True,            # none are a client's own asset
        'snapshot': snapshot,
        'methane_excess_pct': excess_value,   # interpreted consistently per method
        'flaring_bcm_yr': flaring,
        'trajectory': trajectory,
        'verdict': 'performant' if excess_value < 0.3 else 'progress',
        'source': 'TROPOMI S5P CH4 / NO2 / CO; VIIRS Nightfire 2024',
        'generated': datetime.date.today().isoformat(),
        'note': 'Public satellite data. Reference/benchmark sites, not a specific operator\'s assets.'
    }

facilities = [
    package('Groundbirch', 'Shell plc', 'Montney, Canada', 'facility',
            groundbirch_result if 'groundbirch_result' in dir() else result,
            trajectory, flaring_bcm(55.962807, -121.077735),
            'methane_excess_pct', -0.109),
    package('Korpezhe', 'State (Turkmengaz)', 'W. Turkmenistan', 'facility',
            korpezhe, korpezhe_trajectory, korpezhe_flaring,
            'methane_excess_pct', 0.624),
    package('Permian Basin (Delaware)', 'Multiple operators', 'Texas/New Mexico, USA', 'basin',
            permian, permian_trajectory, permian_flaring,
            'enhancement_pct', 2.411),
]

with open('facilities.json', 'w') as f:
    json.dump(facilities, f, indent=2)

print('Wrote facilities.json with', len(facilities), 'facilities ✓')
print(json.dumps(facilities[0], indent=2)[:500], '...')

Wrote facilities.json with 3 facilities ✓
{
  "name": "Groundbirch",
  "operator": "Shell plc",
  "region": "Montney, Canada",
  "method": "facility",
  "is_reference_site": true,
  "snapshot": {
    "lat": 55.962807,
    "lon": -121.077735,
    "methane_site_ppb": 1862.95,
    "methane_bkgd_ppb": 1864.99,
    "methane_excess_pct": -0.109,
    "no2_mol_m2": 1.1528536208611025e-05,
    "co_mol_m2": 0.02796838491574994
  },
  "methane_excess_pct": -0.109,
  "flaring_bcm_yr": 0.0,
  "trajectory": [
    {
      "month": "2025-05",
      "me ...
